# 05. Sectors: GICS through the link, month by month

**What this notebook does.** It gives every universe member in every month its company's GVKEY through the link, and the GICS classification in force on the last trading day of the month before. It counts by year the member-months left Unclassified, and why, judges the coverage condition that sets where the evaluation window starts, and computes by month the weight of each GICS sector in the benchmark (the universe with each member weighted by its market value at the start of the month).

**Why it comes here.** The mandate built later holds each sector within 2 percentage points of its weight in the benchmark, cuts two sectors to half weight and excludes two sub-industries, and the attribution (the split of the portfolio's return minus the benchmark's into parts, sector by sector) reads the same sectors. Each needs a sector for every member-month, read as it stood when the month began.

**Terms used in this notebook.**

- *universe*: the stocks in CRSP's list of index members on the last trading day of the month before.
- *PERMNO*: CRSP's permanent number for a security. *GVKEY*: Compustat's permanent number for a company.
- *link*: CRSP's table of which GVKEY belongs to which PERMNO, and over which dates, each link with a type and a primary flag. For the sector join a member takes the GVKEY of the link of type LC or LU valid on the last trading day of the month before, whatever its primary flag. Where two are valid, the flags are taken in the order P, C, J, N, then the link that started later, and the member-months concerned are counted. The flag says which of a company's securities is its primary one, which matters going from a company to a security and not from a security to its company. A second share class, such as a company's non-voting shares, reaches its company through a link flagged J, and it has its company's sector.
- *GICS*: the Global Industry Classification Standard of S&P and MSCI. It places a company in one of 11 sectors (a two-digit code), and below them in an industry group (four digits), an industry (six) and a sub-industry (eight). Compustat keeps each company's classifications with the dates each was in force.
- *Unclassified*: the twelfth sector, holding every member-month with no GVKEY through a link or no GICS classification in force. It is carried through the mandate and the attribution like the other eleven, so that every sum over sectors still adds up.
- *market capitalisation*: price times shares outstanding, at the end of the month before.
- *benchmark*: the universe weighted by market capitalisation at the start of the month.
- *sector weight*: the sum of the benchmark's weights of the members in a sector. In memory the twelve weights of a month add up to one. The committed table holds the eleven GICS sectors as shares of the classified capitalisation, from July 1999 to December 2024.
- *cell rule*: fixed in `src/slt/constants.py` on 11 November 2026 for every table by month. A cell is committed only if it sums at least three members and no member is more than 80 per cent of it, so that no figure by month is one security's weight under another name. A sector with no member in a month has weight zero and is committed as zero. Unclassified, which holds few members, is never committed by month.
- *mandate*: the full set of constraints the portfolio of the later notebooks obeys.
- *evaluation window*: the months over which the later notebooks run and judge the mandate, from July 1999 to December 2024, 306 months.
- *coverage condition*: fixed in `src/slt/constants.py` on 11 November 2026, before the coverage was seen. In the window's first month and in every later month, at least 95 per cent of the universe's members and at least 95 per cent of its capitalisation have a GICS sector. If July 1999 misses it, the window starts in the first later month of 1999 that meets it. If no month of 1999 meets it, or a later month falls below, the study stops for a decision on the window.

**What I expect to see, and where it comes from.** From the manifest and structural counts of 10 November 2026: Compustat's GICS history has 45,996 rows, pulled on 10 November, and its earliest start date is 30 June 1999. GICS was introduced in 1999. So every member-month before July 1999 is expected Unclassified. From July 1999 the share of member-months with a sector is expected close to one, below one where a member has no GVKEY through a link of type LC or LU or a gap between two of its classifications. Some members are expected to be classified only through a link flagged J, second share classes in the main, and they are counted by year. The member-months with more than one valid link, and with two classifications in force on one day, are expected close to zero, and both are printed. Each month's weights add up to one by construction, which `tests/test_sectors_validation.py` tests on cases worked by hand.

The evaluation window starts in July 1999 for that reason: before it, every member would sit in Unclassified, and the mandate's sector limits, its cut of two sectors to half weight and its two exclusions could not bind. The coverage condition is expected to hold from July 1999, because the GICS history starts with a classification for the companies in the index on 30 June 1999. The eleven GICS sectors are expected to pass the cell rule in every month of the window, and Unclassified to fail it in most. The data layer's gates and its usable window do not depend on sectors and are unaffected.

Running time: about a minute.

In [ ]:
import json
import pathlib
import sys
import time

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from slt import constants as C
from slt import extraction as X

root = X.data_root()             # stops with one sentence when the data root is not set
manifest = X.check_manifest(root)   # stops when the manifest on disk is not the one the constants fix
print("manifest SHA-256", C.MANIFEST_SHA256[:12], "| files", manifest["counts"]["files"],
      "| rows", f"{manifest['counts']['rows']:,}")

In [ ]:
from slt import layer as L
from slt import returns as R
from slt import sectors as S
from slt import universe as U
from slt import validation as V

## 1. The link and the GICS history

Both tables are read whole. The link keeps the links of type LC or LU with a PERMNO, whatever their primary flag. The GICS history is kept as it is: each row is one classification of one company, with its start and end dates.

In [ ]:
t0 = time.time()
universe = L.load_universe(root)
data = L.load_panel(root, universe)
sectors = L.load_sectors(root, data["panel"])
links, history, classified = sectors["links"], sectors["history"], sectors["classified"]
print(f"link rows {sectors['link_rows']:,} | links of types LC and LU with a PERMNO {len(links):,}",
      "| by primary flag", links['linkprim'].value_counts().to_dict())
print("member-months with more than one valid link:", sectors["multiple_gvkeys"])
print(f"GICS rows {len(history):,} | companies {history['gvkey'].nunique():,} | earliest start {history['indfrom'].min().date()}"
      f" | classification types {history['indtype'].value_counts().to_dict()}")
print("member-months with two GICS classifications in force on one day:", sectors["overlapping_gics"])
print("date values present that did not parse:", {**universe["unparsed"], **data["unparsed"], **sectors["unparsed"]} or "none")
print(f"{time.time() - t0:.0f} s")

## 2. Coverage by year, and the Unclassified member-months by cause

By year: the member-months, those with a GVKEY, those with a sector, those with a sector through a link flagged J or N, and the Unclassified ones split by cause. A member-month is Unclassified when its member has no GVKEY through a link, or has one with no classification in force on the day. Of the second kind, those whose company has a classification that starts later are counted: applying a later classification to earlier months would fill them, which the study does not do, because that classification was not known when the month began.

In [ ]:
causes = S.later_classification_by_year(classified, history).drop(columns="months_unclassified")
coverage = S.coverage_by_year(classified).merge(causes, on="year")
has_sector = (classified["sector"] != C.UNCLASSIFIED).values
first_sector = classified.loc[has_sector, "month"].min() if has_sector.any() else None
print("first month with a sector:", first_sector)
print(coverage[coverage["year"] >= 1990].to_string(index=False))

## 3. The evaluation window and its coverage condition

By month, the share of the universe's members with a GICS sector and the share of its capitalisation held by members with one, members without a capitalisation left out of the second. The condition is judged from July 1999: the window starts in the first month of 1999 from July that meets both levels, and holds if every later month to December 2024 meets them too. The shares are printed by month for 1999. The committed table gives them by year, the row for 1999 holding July to December.

In [ ]:
by_month = S.coverage_by_month(classified)
first_year = pd.Period(C.EVAL_START, "M").year
print(by_month[by_month.index.year == first_year].round(6).to_string())
evaluation = S.evaluation_start(by_month)
evaluation["first_month_with_a_sector"] = str(first_sector)
start = evaluation["start"] or C.EVAL_START
after = S.coverage_by_year_after(by_month, start=start)
print(after.to_string(index=False))
print(json.dumps(evaluation, indent=1))
start_text = (f"The window starts in {evaluation['start']} and runs {evaluation['months']} months to {evaluation['end']}"
              if evaluation["start"] else f"No month of {first_year} from {C.EVAL_START} meets both levels")
print(V.verdict_line("The coverage condition", evaluation["condition_met"],
      f"{start_text}, with {evaluation['months_below_after_start']} later months below a level, against levels of "
      f"{C.EVAL_GICS_MEMBER_SHARE_MIN:.0%} of members and {C.EVAL_GICS_CAP_SHARE_MIN:.0%} of capitalisation."
      + ("" if evaluation["condition_met"] else " The study stops here for a decision on the window.")))

## 4. The benchmark's sector weights, and the cell rule

In memory, the twelve sector weights of each month, Unclassified included, as shares of the universe's capitalisation. Members without a capitalisation have no weight, as in notebook 04. For the committed table, the eleven GICS sectors as shares of the classified capitalisation, from the window's first month to December 2024. Before anything is written, every one of its cells is checked against the cell rule, and the notebook stops if one fails, removing any earlier table so that none is left to commit. Unclassified is checked too, for the record: it is never committed by month.

In [ ]:
weights = S.sector_weights(classified)
last = pd.Period(C.EVAL_END, "M")
committed_weights = S.sector_weights(classified, classified_only=True)
committed_weights = committed_weights[(committed_weights["month"] >= pd.Period(start, "M"))
                                      & (committed_weights["month"] <= last)].reset_index(drop=True)
gap = (committed_weights[S.gics_columns()].sum(axis=1) - 1).abs().max()
print(f"months {len(committed_weights)} | largest gap of a month's eleven weights from one: {gap:.2e}")
cells = S.cells_by_month(classified)
rule = S.cell_rule(cells, C.GICS_SECTORS, start, C.EVAL_END)
unclassified_rule = S.cell_rule(cells, [C.UNCLASSIFIED], start, C.EVAL_END)
print("the eleven GICS sectors:", json.dumps(rule))
print("Unclassified, not committed by month:", json.dumps(unclassified_rule))
print(V.verdict_line("The cell rule", rule["rule_met"],
      f"{rule['cells_failing']} of {rule['cells']} cells fail, against at least {C.SMALL_CELL_MIN_MEMBERS} members and no member "
      f"above {C.SMALL_CELL_MAX_MEMBER_SHARE:.0%} of a cell; {rule['empty_cells']} cells are empty and committed as zero."))
if not rule["rule_met"]:
    (ROOT / C.RESULTS_DIR / "sector_weights_monthly.csv").unlink(missing_ok=True)
    raise SystemExit("A cell of the table by month fails the cell rule, so the table is not written and the study stops here.")
names = {"weight_" + code: name for code, name in C.GICS_SECTORS.items()}
names["weight_" + C.UNCLASSIFIED.lower()] = C.UNCLASSIFIED
shown = [m for m in (start, "2007-12", "2024-12") if pd.Period(m, "M") in set(weights["month"])]
table = weights.assign(month=weights["month"].astype(str)).set_index("month").loc[shown, S.sector_columns()].rename(columns=names)
print("In memory, twelve sectors as shares of the universe's capitalisation:")
print(table.T.round(4).to_string())

## 5. What is committed

Four files go into `results/20260928/`, each an aggregate with no identifier of a security: the coverage by year with the Unclassified member-months by cause, the coverage condition's two shares by year from the window's first month, the eleven GICS sector weights by month over the window, every cell of which has passed the cell rule, and a summary of the counts printed in this notebook, the cell rule's figures and the evaluation window's start.

In [ ]:
weights_out = committed_weights.assign(month=committed_weights["month"].astype(str))
summary = {
    "written_by": "notebooks/05_sectors.ipynb", "manifest_sha256": C.MANIFEST_SHA256, "gics_timing": C.GICS_TIMING,
    "sector_link_flags": list(C.SECTOR_LINK_FLAGS),
    "link_rows": sectors["link_rows"], "links_kept": int(len(links)),
    "member_months_with_more_than_one_valid_link": int(sectors["multiple_gvkeys"]),
    "gics_rows": int(len(history)), "member_months_with_two_classifications": int(sectors["overlapping_gics"]),
    "evaluation_window": evaluation,
    "cell_rule": {"min_members": C.SMALL_CELL_MIN_MEMBERS, "max_member_share": C.SMALL_CELL_MAX_MEMBER_SHARE,
                  "gics_sectors": rule, "unclassified_not_committed": unclassified_rule},
    "date_values_not_parsed": {**universe["unparsed"], **data["unparsed"], **sectors["unparsed"]},
}
written = [X.write_results("sector_coverage_by_year.csv", frame=coverage),
           X.write_results("evaluation_coverage_by_year.csv", frame=after),
           X.write_results("sector_weights_monthly.csv", frame=weights_out),
           X.write_results("sectors_summary.json", record=summary)]
for path in written:
    print("written:", path.relative_to(ROOT))

## What was learned, and what could be wrong

What the notebook establishes: how many member-months have a sector, by year, how many reach it only through a link flagged J or N, why the others have none, the first month with any sector, whether the coverage condition holds and so where the evaluation window starts, and the eleven GICS sector weights by month, each cell checked against the cell rule. There is no gate or expectation on sectors in the data layer.

What could still be wrong. A sector read through the link depends on the link being right on the day, and a link that points at the wrong company would give a member another company's sector without any count here showing it. Compustat's history records a classification from the date it was assigned, so a company reclassified in a GICS revision, such as the one of 2018 that created Communication Services, changes sector on that date and not before, which is the intended point-in-time reading. And a member with no classification sits in Unclassified with its full weight. Before July 1999 that is every member, which is why the evaluation window starts there.